In [1]:
import pandas as pd
import google.generativeai as genai
import os
import re
import time

# --- Configuration ---
API_KEY_FILE = "apikey.txt"
CSV_FILE = "marketing.csv"
# Consider 'gemini-1.5-flash-latest' for faster, cost-effective processing if suitable
MODEL_NAME = 'gemini-2.5-flash-preview-05-20'
# MODEL_NAME = 'gemini-1.5-pro-latest' # Potentially more accurate, but slower/costlier
# MODEL_NAME = 'gemini-1.5-flash-latest' # Faster and cheaper, good for bulk tasks

# --- Helper Functions ---

def get_api_key(filepath="apikey.txt"):
    """Reads the API key from the specified file."""
    try:
        with open(filepath, "r") as f:
            return f.read().strip()
    except FileNotFoundError:
        print(f"エラー: APIキーファイル '{filepath}' が見つかりません。")
        return None

def get_llm_choice(prompt_text, model, question_number):
    """
    Gets the LLM's choice (A, B, C, or D) for a given prompt.

    Args:
        prompt_text (str): The prompt to send to the LLM.
        model (genai.GenerativeModel): The initialized Gemini model.
        question_number (int): The current question number for logging.

    Returns:
        str or None: The LLM's choice ('A', 'B', 'C', 'D') or None if parsing fails.
    """
    max_retries = 3
    retry_delay = 5 # seconds

    for attempt in range(max_retries):
        try:
            print(f"\nQ{question_number}: LLMに問い合わせ中...")
            response = model.generate_content(prompt_text)
            raw_text = response.text.strip()

            match = re.search(r"\b([A-D])\b", raw_text.upper())
            if match:
                choice = match.group(1)
                print(f"  LLMの生回答: \"{raw_text}\" -> 解析結果: {choice}")
                return choice
            else:
                if len(raw_text) == 1 and raw_text.upper() in ["A", "B", "C", "D"]:
                    choice = raw_text.upper()
                    print(f"  LLMの生回答: \"{raw_text}\" -> 解析結果: {choice}")
                    return choice

                print(f"  警告 (Q{question_number}): LLMの回答を解析できませんでした。回答: \"{raw_text}\"")
                return None

        except Exception as e:
            print(f"  エラー (Q{question_number}, 試行 {attempt + 1}/{max_retries}): LLM呼び出し中にエラーが発生しました: {e}")
            if "rate limit" in str(e).lower() and attempt < max_retries - 1:
                print(f"    レート制限の可能性。{retry_delay * (attempt + 1)}秒後に再試行します...")
                time.sleep(retry_delay * (attempt + 1))
            elif attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                print(f"  LLM呼び出しの最大再試行回数に達しました (Q{question_number})。")
                return None
    return None


def evaluate_marketing_questions(csv_filepath, api_key):
    """
    Reads marketing questions from a CSV, gets LLM answers, and calculates accuracy.
    """
    if not api_key:
        print("APIキーが提供されていないため、処理を中止します。")
        return

    genai.configure(api_key=api_key)
    try:
        model = genai.GenerativeModel(MODEL_NAME)
    except Exception as e:
        print(f"モデルの初期化中にエラーが発生しました ({MODEL_NAME}): {e}")
        return

    required_columns = ['問題', '選択肢A', '選択肢B', '選択肢C', '選択肢D', '正解']

    try:
        # CSVファイルにヘッダー行がないことを想定し、カラム名を指定して読み込む
        df = pd.read_csv(csv_filepath, header=None, names=required_columns, encoding='utf-8-sig')

        # DataFrameに指定したカラム名が正しく設定されたか念のため確認
        if not all(col in df.columns for col in required_columns):
            print(f"エラー: CSVファイルのカラム名設定に問題があります。")
            print(f"期待されたカラム: {required_columns}")
            print(f"現在のDataFrameカラム: {df.columns.tolist()}")
            return
            
    except FileNotFoundError:
        print(f"エラー: CSVファイル '{csv_filepath}' が見つかりません。")
        return
    except pd.errors.EmptyDataError:
        print(f"エラー: CSVファイル '{csv_filepath}' は空です。")
        return
    except Exception as e:
        print(f"CSVファイルの読み込み中に予期せぬエラーが発生しました: {e}")
        return

    total_questions = 0
    correct_answers = 0

    if df.empty:
        print(f"CSVファイル '{csv_filepath}' にデータがありません。")
        return

    for index, row in df.iterrows():
        total_questions += 1
        question_number = index + 1
        try:
            problem = str(row['問題']) # 文字列に変換
            option_a = str(row['選択肢A'])
            option_b = str(row['選択肢B'])
            option_c = str(row['選択肢C'])
            option_d = str(row['選択肢D'])
            correct_solution = str(row['正解']).strip().upper()
        except KeyError as e:
            print(f"エラー (Q{question_number}): CSVの行にキーエラーがあります: {e}。この行をスキップします。")
            continue

        if not all([problem, option_a, option_b, option_c, option_d, correct_solution]):
            print(f"警告 (Q{question_number}): データが不足している行があります。問題: '{problem}', 正解: '{correct_solution}'。この行をスキップします。")
            continue

        if correct_solution not in ["A", "B", "C", "D"]:
            print(f"警告 (Q{question_number}): 正解の形式が不正です ('{correct_solution}')。期待されるのはA, B, C, Dのいずれかです。この行をスキップします。")
            continue

        prompt = f"""以下のマーケティングに関する問題に、最も適切な選択肢をA、B、C、Dのいずれか一つで答えてください。記号のみを回答してください。

問題: {problem}
A: {option_a}
B: {option_b}
C: {option_c}
D: {option_d}

あなたの答え (A, B, C, D のみ): """

        llm_answer = get_llm_choice(prompt, model, question_number)

        if llm_answer:
            print(f"  Q{question_number}: LLMの回答 = {llm_answer}, 正解 = {correct_solution}")
            if llm_answer == correct_solution:
                correct_answers += 1
                print("    結果: 正解")
            else:
                print("    結果: 不正解")
        else:
            print(f"  Q{question_number}: LLMから有効な回答を得られませんでした。この問題は不正解として扱います。正解 = {correct_solution}")

        time.sleep(1.2)

    if total_questions > 0:
        accuracy = (correct_answers / total_questions) * 100
        print(f"\n--- 結果 ---")
        print(f"総問題数: {total_questions}")
        print(f"正解数: {correct_answers}")
        print(f"正解率: {accuracy:.2f}%")
    else:
        print("\n評価対象となる有効な問題がCSVファイルにありませんでした。")

# --- Main Execution ---

if __name__ == "__main__":
    api_key = get_api_key(API_KEY_FILE)

    if not os.path.exists(CSV_FILE):
        print(f"'{CSV_FILE}' が見つかりません。処理を続行するには、正しいCSVファイルを用意してください。")
        # ダミーデータ作成ロジックは必要に応じてここに再挿入できますが、
        # 通常はユーザーが実際のデータファイルを用意することを前提とします。
        # print(f"'{CSV_FILE}' が見つかりません。ダミーファイルを作成します...")
        # dummy_data = {
        #     '問題': [
        #         "マーケティングの4Pとは何ですか？", "SWOT分析の「O」は何を表しますか？"
        #     ],
        #     '選択肢A': ["製品・価格・場所・販促", "機会 (Opportunities)"],
        #     '選択肢B': ["製品・人物・場所・過程", "脅威 (Threats)"],
        #     '選択肢C': ["価格・計画・人々・販促", "強み (Strengths)"],
        #     '選択肢D': ["製品・価格・過程・人々", "弱み (Weaknesses)"],
        #     '正解': ["A", "A"]
        # }
        # dummy_df = pd.DataFrame(dummy_data)
        # try:
        #     dummy_df.to_csv(CSV_FILE, index=False, header=False, encoding='utf-8-sig') # header=False for no header
        #     print(f"ダミーファイル '{CSV_FILE}' をヘッダーなしで作成しました。")
        # except Exception as e:
        #     print(f"ダミーファイルの作成に失敗しました: {e}")

    if api_key and os.path.exists(CSV_FILE): # CSVファイルの存在も確認
        evaluate_marketing_questions(CSV_FILE, api_key)
    elif not api_key:
        print("処理を開始できません。APIキーを確認してください。")
    elif not os.path.exists(CSV_FILE):
        print(f"処理を開始できません。CSVファイル '{CSV_FILE}' を確認してください。")

c:\Users\eriya\anaconda3\envs\python-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Q1: LLMに問い合わせ中...
  LLMの生回答: "A" -> 解析結果: A
  Q1: LLMの回答 = A, 正解 = A
    結果: 正解

Q2: LLMに問い合わせ中...
  LLMの生回答: "C" -> 解析結果: C
  Q2: LLMの回答 = C, 正解 = C
    結果: 正解

Q3: LLMに問い合わせ中...
  LLMの生回答: "この問題は、製品をサービスでどの程度「包み込む」か（サービス化の度合い）に影響を与える要因について尋ねています。

各選択肢を分析します。

*   **A: 製品の種類に関連する接伴性のレベル。**
    製品の種類（例：複雑な産業機械、家電製品、ソフトウェア）によって、設置、メンテナンス、トレーニング、サポートなどのサービスの必要性が異なります。製品が複雑であるほど、より多くのサービスで「包まれる」傾向があります。これは要因として適切です。

*   **B: パフォーマンスの価値。**
    顧客が製品から得たい「パフォーマンス」や「成果」の価値が高い場合、そのパフォーマンスを保証・最大化するためのサービス（例：予防保全、遠隔監視、アップグレード）が不可欠になります。例えば、「稼働時間」や「生産性」そのものを売る場合、サービスは製品に深く組み込まれます。これは要因として適切です。

*   **C: 需要と供給の変動。**
    需要と供給の変動は、市場価格、販売量、生産計画などに影響を与えますが、製品に付随するサービスの「範囲」や「深さ」といった本質的な部分を決定する要因ではありません。一時的な市場の変動によってサービスの提供方法や価格が調整されることはあっても、製品とサービスの関係性の根本的な度合いを左右するものではありません。

*   **D: サービスの提供方法。**
    サービスがどのように提供されるか（例：サブスクリプションモデル、従量課金、オンサイトサービス、リモートサポート）は、製品をサービスで「包む」度合いに大きく影響します。例えば、「Product-as-a-Service (PaaS)」のような提供方法は、製品の所有権ではなく「利用」や「成果」を提供するため、サービスが製品全体を深く包み込むことになります。これは要因として適切です。

したがって、